# 04 – Feature Engineering và bàn giao dữ liệu cho ba mô hình

**Đầu vào:** `data/processed/imdb_cleaned.csv`.

**Đầu ra:** `imdb_featured.csv` và ba CSV tương ứng PCA/K-Means, Regression, Classification.

Để tránh rò rỉ dữ liệu, bộ phận tiền xử lý **chưa fit bộ điền khuyết / StandardScaler / encoder có học tham số trên toàn bộ dữ liệu**. Các mô hình phải fit chúng trên train (riêng phân cụm có thể fit trên toàn dữ liệu dùng phân cụm).

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'data' / 'processed'
input_path = OUT / 'imdb_cleaned.csv'
if not input_path.exists():
    raise FileNotFoundError('Chạy 02_data_cleaning.ipynb trước!')
df = pd.read_csv(input_path)
print('Đầu vào:', df.shape)

Đầu vào: (3716, 20)


## 4.1. Tạo các thuộc tính mới

In [2]:
# Các biến năm/thời lượng/danh sách
for col in ['year', 'duration_minutes', 'imdb_rating', 'metascore']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['release_decade'] = (np.floor(df['year'] / 10) * 10).astype('Int64')
df['title_length'] = df['title'].fillna('').astype(str).str.len()

def count_items(value, delimiter=r'[,;|]'):
    if pd.isna(value) or str(value).strip().lower() in {'', 'unknown', 'nan'}:
        return np.nan
    return len([x for x in re.split(delimiter, str(value)) if x.strip()])

df['genre_count'] = df['genres'].apply(count_items)
df['cast_count'] = df['star_cast'].apply(count_items)
df['primary_genre'] = df['genres'].astype('string').str.split(',').str[0].str.strip()
df.loc[df['primary_genre'].eq('Unknown'), 'primary_genre'] = pd.NA

df['duration_group'] = pd.cut(df['duration_minutes'],
                              bins=[0, 90, 120, np.inf],
                              labels=['Short (<=90)', 'Medium (91-120)', 'Long (>120)'])

# Nhãn phân loại sinh ra từ điểm IMDb và sẽ KHÔNG được dùng làm feature.
df['rating_class'] = pd.Series(pd.NA, index=df.index, dtype='string')
valid_rating = df['imdb_rating'].notna()
df.loc[valid_rating, 'rating_class'] = np.where(df.loc[valid_rating, 'imdb_rating'] >= 7,
                                                 'High', 'Low')

show_cols = ['title', 'year', 'release_decade', 'duration_minutes',
             'duration_group', 'primary_genre', 'genre_count',
             'imdb_rating', 'rating_class']
display(df[show_cols].head(10))

,title,year,release_decade,duration_minutes,duration_group,primary_genre,genre_count,imdb_rating,rating_class
0,End of the Spear,2005,2000,108.0,Medium (91-120),Adventure,1,6.8,Low
1,Elvira Madigan,1967,1960,91.0,Medium (91-120),Biography,1,7.0,High
2,The Kid Stays in the Picture,2002,2000,93.0,Medium (91-120),Documentary,1,7.3,High
3,It Ain't Over,2022,2020,99.0,Medium (91-120),Documentary,1,8.2,High
4,Mahler,1974,1970,115.0,Medium (91-120),Biography,1,7.0,High
5,The Dark Horse,2014,2010,124.0,Long (>120),Biography,1,7.4,High
6,Stephen Curry: Underrated,2023,2020,110.0,Medium (91-120),Documentary,1,7.3,High
7,Carrington,1995,1990,121.0,Long (>120),Biography,1,6.8,Low
8,Burden,2018,2010,117.0,Medium (91-120),Biography,1,6.7,Low
9,Georgetown,2019,2010,99.0,Medium (91-120),Biography,1,6.2,Low


## 4.2. Mã hóa đa nhãn thể loại theo danh sách thể loại CỐ ĐỊNH

Không chọn top-k theo toàn tập để tránh thông tin về tập kiểm tra ảnh hưởng đến thiết kế đặc trưng. Mã hóa còn lại (ví dụ `primary_genre`) sẽ làm bằng `OneHotEncoder` trong pipeline huấn luyện.

In [3]:
COMMON_GENRES = ['Action','Adventure','Animation','Biography','Comedy','Crime',
                 'Documentary','Drama','Family','Fantasy','History','Horror',
                 'Music','Musical','Mystery','Romance','Sci-Fi','Sport',
                 'Thriller','War','Western']

def has_genre(value, genre):
    if pd.isna(value):
        return 0
    tokens = [x.strip().casefold() for x in str(value).split(',')]
    return int(genre.casefold() in tokens)

for genre in COMMON_GENRES:
    name = 'genre_' + genre.lower().replace('-', '_')
    df[name] = df['genres'].apply(lambda value, label=genre: has_genre(value, label)).astype('int8')

genre_binary_cols = ['genre_' + g.lower().replace('-', '_') for g in COMMON_GENRES]
print('Các cột thể loại nhị phân:', genre_binary_cols)
display(df[['title'] + genre_binary_cols[:8]].head())

Các cột thể loại nhị phân: ['genre_action', 'genre_adventure', 'genre_animation', 'genre_biography', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_musical', 'genre_mystery', 'genre_romance', 'genre_sci_fi', 'genre_sport', 'genre_thriller', 'genre_war', 'genre_western']


,title,genre_action,genre_adventure,genre_animation,genre_biography,genre_comedy,genre_crime,genre_documentary,genre_drama
0,End of the Spear,0,1,0,0,0,0,0,0
1,Elvira Madigan,0,0,0,1,0,0,0,0
2,The Kid Stays in the Picture,0,0,0,0,0,0,1,0
3,It Ain't Over,0,0,0,0,0,0,1,0
4,Mahler,0,0,0,1,0,0,0,0


## 4.3. Tạo 3 bộ dữ liệu và tránh rò rỉ mục tiêu

- **Regression:** dự đoán `imdb_rating`; không đưa `rating_class` vào đặc trưng.
- **Classification:** dự đoán `rating_class` = High/Low; **không đưa `imdb_rating` hay biến bắt nguồn trực tiếp từ nó** vào đặc trưng.
- **PCA / K-Means:** phân cụm dựa trên dữ liệu phim, chưa chuẩn hóa tại bước xuất CSV.

In [4]:
feature_num = ['year', 'duration_minutes', 'metascore', 'genre_count',
               'cast_count', 'title_length']
feature_cat = ['primary_genre', 'certificates']
features = feature_num + genre_binary_cols + feature_cat

# Tập hồi quy: nhãn số cần có giá trị thực.
regression = df.loc[df['imdb_rating'].notna(),
                    ['title'] + features + ['imdb_rating']].copy()

# Tập phân loại: KHÔNG có cột imdb_rating để tránh label leakage.
classification = df.loc[df['rating_class'].notna(),
                        ['title'] + features + ['rating_class']].copy()

# Tập phân cụm: tên phim dùng để trình bày, cột số/cờ thể loại dùng làm đặc trưng.
cluster_features = ['year', 'duration_minutes', 'metascore',
                    'imdb_rating', 'genre_count', 'cast_count'] + genre_binary_cols
clustering = df[['title'] + cluster_features].copy()

assert 'imdb_rating' not in classification.columns
assert 'rating_class' not in regression.columns
assert regression['imdb_rating'].notna().all()
assert classification['rating_class'].notna().all()
print('Regression:', regression.shape)
print('Classification:', classification.shape)
print('Clustering:', clustering.shape)
print('\nNhãn Classification:')
print(classification['rating_class'].value_counts())

Regression: (3716, 31)
Classification: (3716, 31)
Clustering: (3716, 28)

Nhãn Classification:
rating_class
Low     2090
High    1626
Name: count, dtype: Int64


## 4.4. Quy tắc chia tập dữ liệu và chuẩn hóa

Các CSV ở trên giữ NaN có kiểm soát để các thành viên mô hình xây **pipeline**: chia train/test → điền missing từ train → StandardScaler số, OneHotEncoder biến phân loại → fit model. Đoạn dưới minh họa bằng một lần chia hồi quy, không lưu bộ scaler đã fit vào file chung.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

if len(regression) >= 5:
    X = regression[features]
    y = regression['imdb_rating']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    numeric_transform = Pipeline([
        ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
        ('scaler', StandardScaler()),
    ])
    category_transform = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ])
    preprocessing = ColumnTransformer([
        ('numerical', numeric_transform, feature_num + genre_binary_cols),
        ('categorical', category_transform, feature_cat),
    ])
    X_train_scaled = preprocessing.fit_transform(X_train)
    X_test_scaled = preprocessing.transform(X_test)
    print('Train:', X_train.shape, '| Test:', X_test.shape)
    print('Số feature sau biến đổi:', X_train_scaled.shape[1])
else:
    print('Không đủ 5 phim có điểm IMDb để minh họa train/test.')

Train: (2972, 29) | Test: (744, 29)
Số feature sau biến đổi: 62


## 4.5. Lưu kết quả và kiểm tra lần cuối

In [6]:
exports = {
    'imdb_featured.csv': df,
    'imdb_for_regression.csv': regression,
    'imdb_for_classification.csv': classification,
    'imdb_for_clustering.csv': clustering,
}
for filename, frame in exports.items():
    output = OUT / filename
    frame.to_csv(output, index=False, encoding='utf-8-sig')
    print(f'{filename}: {frame.shape[0]:,} dòng × {frame.shape[1]} cột')

assert df['title'].notna().all()
assert regression['imdb_rating'].between(0,10).all()
assert classification['rating_class'].isin(['High','Low']).all()
print('HOÀN TẤT. File imdb_cleaned.csv từ notebook 02 được giữ nguyên.')

imdb_featured.csv: 3,716 dòng × 48 cột
imdb_for_regression.csv: 3,716 dòng × 31 cột
imdb_for_classification.csv: 3,716 dòng × 31 cột
imdb_for_clustering.csv: 3,716 dòng × 28 cột
HOÀN TẤT. File imdb_cleaned.csv từ notebook 02 được giữ nguyên.
